In [ ]:
"""
Source 1: MBTA LAMP subway on-time performance (historical file download)

Portal: https://performancedata.mbta.com/
Dictionary: https://github.com/mbta/lamp/blob/main/Data_Dictionary.md

One Parquet file per service date:
  https://performancedata.mbta.com/lamp/subway-on-time-performance-v1/YYYY-MM-DD-subway-on-time-performance-v1.parquet
An index of every published date lives at .../index.csv

Each row is one trip_id-stop_id pair: a single train's visit to a single
platform, with the scheduled time and the observed time.

THE UNIT TRAP (read this before trusting any delay number)
    scheduled_arrival_time / scheduled_departure_time
        int64, SECONDS AFTER MIDNIGHT of service_date, local service day.
        Values can exceed 86400 for after-midnight service.
    stop_timestamp / move_timestamp
        int64, UNIX EPOCH SECONDS (UTC).
    These are different clocks in different units. Subtracting them raw gives
    nonsense. `add_delay_columns` converts the schedule to epoch first.

OTHER GOTCHAS
    - `parent_station` holds the station NAME, not an ID. There is no
      stop_name column. This is the field you join to Wikipedia / gated entries.
    - `stop_timestamp` is the observed "STOPPED_AT" time, but falls back to the
      last predicted arrival from StopTimeUpdate when the vehicle feed is
      missing -- and there is no flag saying which was used.
    - There is no actual-departure column. Departure from stop N is the
      `move_timestamp` of stop N+1, or `stop_timestamp + dwell_time_seconds`.
    - `branch_route_id` is NULL for single-route lines (Blue, Orange). LAMP
      inserts Red-A / Red-B for Red Line trips south of JFK/UMass.
"""

from __future__ import annotations

import io

import pandas as pd
import requests

BASE = "https://performancedata.mbta.com/lamp/subway-on-time-performance-v1"
INDEX_URL = f"{BASE}/index.csv"
SERVICE_TZ = "America/New_York"

HEADERS = {"User-Agent": "transit-delay-weather-class-project/1.0"}


def list_service_dates() -> pd.DataFrame:
    """Every published service date. Use this to pick a real date range."""
    resp = requests.get(INDEX_URL, headers=HEADERS, timeout=60)
    resp.raise_for_status()

    df = pd.read_csv(io.BytesIO(resp.content))
    df.columns = [c.strip().lower() for c in df.columns]

    date_col = next((c for c in df.columns if "date" in c), df.columns[0])
    df["service_date"] = pd.to_datetime(df[date_col], errors="coerce")
    return df.dropna(subset=["service_date"]).sort_values("service_date")


def fetch_service_date(date: str) -> pd.DataFrame:
    """One service date. `date` is 'YYYY-MM-DD'."""
    url = f"{BASE}/{date}-subway-on-time-performance-v1.parquet"
    resp = requests.get(url, headers=HEADERS, timeout=180)
    resp.raise_for_status()
    return pd.read_parquet(io.BytesIO(resp.content))


def add_delay_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Put schedule and observation on the same clock, then difference them.

    Adds:
        service_day          midnight of the service date (tz-aware, local)
        scheduled_arrival    tz-aware local timestamp
        actual_arrival       tz-aware local timestamp
        arrival_delay_min    actual - scheduled, in minutes (+ = late)
        arrival_hour_local   hour of the local service day, for the weather join

    DST caveat: GTFS seconds-after-midnight are defined against "noon minus
    12 hours", not against literal local midnight. On the two DST changeover
    days per year, times after the transition shift by an hour. Negligible for
    most analysis; drop those two dates if you want to be strict.
    """
    out = df.copy()

    # service_date arrives as int64 YYYYMMDD.
    service_day = pd.to_datetime(
        out["service_date"].astype("int64").astype(str), format="%Y%m%d"
    ).dt.tz_localize(SERVICE_TZ, nonexistent="shift_forward", ambiguous="NaT")
    out["service_day"] = service_day

    for src, name in [
        ("scheduled_arrival_time", "scheduled_arrival"),
        ("scheduled_departure_time", "scheduled_departure"),
    ]:
        if src in out.columns:
            out[name] = service_day + pd.to_timedelta(out[src], unit="s")

    # Observed times are plain unix epoch seconds.
    for src, name in [
        ("stop_timestamp", "actual_arrival"),
        ("move_timestamp", "approach_time"),
    ]:
        if src in out.columns:
            out[name] = (
                pd.to_datetime(out[src], unit="s", utc=True, errors="coerce")
                .dt.tz_convert(SERVICE_TZ)
            )

    out["arrival_delay_min"] = (
        out["actual_arrival"] - out["scheduled_arrival"]
    ).dt.total_seconds() / 60

    out["arrival_hour_local"] = out["actual_arrival"].dt.floor("h")

    return out


def fetch_dates(dates: list[str], with_delay: bool = True) -> pd.DataFrame:
    """Stack several service dates into one DataFrame."""
    frames = []
    for date in dates:
        part = fetch_service_date(date)

    """One service date. `date` is 'YYYY-MM-DD'."""
    url = f"{BASE}/{date}-subway-on-time-performance-v1.parquet"
    resp = requests.get(url, headers=HEADERS, timeout=180)
    resp.raise_for_status()
    return pd.read_parquet(io.BytesIO(resp.content))






In [20]:

index = list_service_dates()
print(f"published dates: {len(index):,}")
print(f"range: {index['service_date'].min().date()} .. {index['service_date'].max().date()}")

# Grab a recent date that definitely exists rather than guessing one.
latest = index["service_date"].iloc[-2].strftime("%Y-%m-%d")
df = fetch_dates([latest])

print(f"\nshape: {df.shape}")
print(
    df[
        [
            "parent_station",
            "trunk_route_id",
            "direction_destination",
            "scheduled_arrival",
            "actual_arrival",
            "arrival_delay_min",
        ]
    ]
    .head(15)
    .to_string()
)
print(f"\nmedian delay (min): {df['arrival_delay_min'].median():.2f}")
print(f"missing actual_arrival: {df['actual_arrival'].isna().sum():,} / {len(df):,}")
print(f"stations: {df['parent_station'].nunique()}")
print([latest])

published dates: 2,572
range: 2019-09-15 .. 2026-09-29
2026-09-28: 36,634 rows

shape: (36634, 34)
   parent_station trunk_route_id direction_destination         scheduled_arrival            actual_arrival  arrival_delay_min
0     place-river          Green          Union Square                       NaT 2026-09-28 03:59:51-04:00                NaN
1     place-river          Green          Union Square                       NaT 2026-09-28 03:59:53-04:00                NaN
2     place-clmnl          Green     Government Center 2026-09-28 15:38:00-04:00 2026-09-28 04:10:54-04:00        -687.100000
3     place-clmnl          Green     Government Center 2026-09-28 11:39:00-04:00 2026-09-28 04:10:54-04:00        -448.100000
4     place-asmnl            Red               Alewife 2026-09-28 05:15:00-04:00 2026-09-28 04:34:43-04:00         -40.283333
5     place-orhte           Blue               Bowdoin 2026-09-28 05:08:00-04:00 2026-09-28 04:36:11-04:00         -31.816667
6     place-asmnl  

In [22]:
df1 = fetch_dates(['2026-09-28'])

2026-09-28: 36,634 rows


In [23]:
df1

,stop_sequence,stop_id,parent_station,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,headway_trunk_seconds,headway_branch_seconds,service_date,...,scheduled_travel_time,scheduled_headway_branch,scheduled_headway_trunk,service_day,scheduled_arrival,scheduled_departure,actual_arrival,approach_time,arrival_delay_min,arrival_hour_local
0,310,70160,place-river,NaN,1.790582e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,NaN,NaN,2026-09-28 00:00:00-04:00,NaT,NaT,2026-09-28 03:59:51-04:00,NaT,NaN,2026-09-28 03:00:00-04:00
1,310,70160,place-river,NaN,1.790582e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,NaN,NaN,2026-09-28 00:00:00-04:00,NaT,NaT,2026-09-28 03:59:53-04:00,NaT,NaN,2026-09-28 03:00:00-04:00
2,180,70238,place-clmnl,NaN,1.790583e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,600.0,600.0,2026-09-28 00:00:00-04:00,2026-09-28 15:38:00-04:00,2026-09-28 15:38:00-04:00,2026-09-28 04:10:54-04:00,NaT,-687.100000,2026-09-28 04:00:00-04:00
3,180,70238,place-clmnl,NaN,1.790583e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,600.0,600.0,2026-09-28 00:00:00-04:00,2026-09-28 11:39:00-04:00,2026-09-28 11:39:00-04:00,2026-09-28 04:10:54-04:00,NaT,-448.100000,2026-09-28 04:00:00-04:00
4,50,70094,place-asmnl,NaN,1.790584e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,NaN,NaN,2026-09-28 00:00:00-04:00,2026-09-28 05:15:00-04:00,2026-09-28 05:15:00-04:00,2026-09-28 04:34:43-04:00,NaT,-40.283333,2026-09-28 04:00:00-04:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36629,310,70161,place-river,NaN,1.790662e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,NaN,NaN,2026-09-28 00:00:00-04:00,NaT,NaT,2026-09-29 02:00:01-04:00,NaT,NaN,2026-09-29 02:00:00-04:00
36630,9,70502,place-lech,1.790662e+09,1.790666e+09,3517.0,NaN,NaN,NaN,20260928,...,240.0,540.0,240.0,2026-09-28 00:00:00-04:00,2026-09-28 17:38:00-04:00,2026-09-28 17:38:00-04:00,2026-09-29 03:09:33-04:00,2026-09-29 02:10:56-04:00,571.550000,2026-09-29 03:00:00-04:00
36631,9,70502,place-lech,1.790663e+09,1.790665e+09,1868.0,NaN,NaN,NaN,20260928,...,240.0,540.0,240.0,2026-09-28 00:00:00-04:00,2026-09-28 17:38:00-04:00,2026-09-28 17:38:00-04:00,2026-09-29 03:02:01-04:00,2026-09-29 02:30:53-04:00,564.016667,2026-09-29 03:00:00-04:00
36632,310,70160,place-river,NaN,1.790667e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,480.0,480.0,2026-09-28 00:00:00-04:00,2026-09-28 12:49:00-04:00,2026-09-28 12:49:00-04:00,2026-09-29 03:37:39-04:00,NaT,888.650000,2026-09-29 03:00:00-04:00


In [ ]:
"""Every published service date. Use this to pick a real date range."""
resp = requests.get(INDEX_URL, headers=HEADERS, timeout=60)
resp.raise_for_status()

df = pd.read_csv(io.BytesIO(resp.content))
df.columns = [c.strip().lower() for c in df.columns]

date_col = next((c for c in df.columns if "date" in c), df.columns[0])
df["service_date"] = pd.to_datetime(df[date_col], errors="coerce")
df = df.dropna(subset=["service_date"]).sort_values("service_date")

In [18]:
df_9_27 = fetch_dates(['2026-09-28', '2026-09-27'])


2026-09-28: 36,634 rows
2026-09-27: 26,308 rows


/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)
/opt/anaconda3/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: overflow encountered in multiply
  return bound(*args, **kwds)


In [19]:
df_9_27

,stop_sequence,stop_id,parent_station,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,headway_trunk_seconds,headway_branch_seconds,service_date,...,scheduled_travel_time,scheduled_headway_branch,scheduled_headway_trunk,service_day,scheduled_arrival,scheduled_departure,actual_arrival,approach_time,arrival_delay_min,arrival_hour_local
0,310,70160,place-river,NaN,1.790582e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,NaN,NaN,2026-09-28 00:00:00-04:00,NaT,NaT,2026-09-28 03:59:51-04:00,NaT,NaN,2026-09-28 03:00:00-04:00
1,310,70160,place-river,NaN,1.790582e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,NaN,NaN,2026-09-28 00:00:00-04:00,NaT,NaT,2026-09-28 03:59:53-04:00,NaT,NaN,2026-09-28 03:00:00-04:00
2,180,70238,place-clmnl,NaN,1.790583e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,600.0,600.0,2026-09-28 00:00:00-04:00,2026-09-28 15:38:00-04:00,2026-09-28 15:38:00-04:00,2026-09-28 04:10:54-04:00,NaT,-687.100000,2026-09-28 04:00:00-04:00
3,180,70238,place-clmnl,NaN,1.790583e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,600.0,600.0,2026-09-28 00:00:00-04:00,2026-09-28 11:39:00-04:00,2026-09-28 11:39:00-04:00,2026-09-28 04:10:54-04:00,NaT,-448.100000,2026-09-28 04:00:00-04:00
4,50,70094,place-asmnl,NaN,1.790584e+09,NaN,NaN,NaN,NaN,20260928,...,NaN,NaN,NaN,2026-09-28 00:00:00-04:00,2026-09-28 05:15:00-04:00,2026-09-28 05:15:00-04:00,2026-09-28 04:34:43-04:00,NaT,-40.283333,2026-09-28 04:00:00-04:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62937,310,70107,place-lake,1.790575e+09,1.790576e+09,255.0,NaN,NaN,NaN,20260927,...,180.0,540.0,540.0,2026-09-27 00:00:00-04:00,2026-09-27 06:15:00-04:00,2026-09-27 06:15:00-04:00,2026-09-28 02:06:12-04:00,2026-09-28 02:01:57-04:00,1191.200000,2026-09-28 02:00:00-04:00
62938,310,70160,place-river,NaN,1.790576e+09,NaN,NaN,NaN,NaN,20260927,...,NaN,NaN,NaN,2026-09-27 00:00:00-04:00,NaT,NaT,2026-09-28 02:07:44-04:00,NaT,NaN,2026-09-28 02:00:00-04:00
62939,310,70160,place-river,NaN,1.790576e+09,NaN,NaN,NaN,NaN,20260927,...,NaN,NaN,NaN,2026-09-27 00:00:00-04:00,NaT,NaT,2026-09-28 02:13:06-04:00,NaT,NaN,2026-09-28 02:00:00-04:00
62940,9,70502,place-lech,1.790577e+09,1.790579e+09,2177.0,NaN,NaN,NaN,20260927,...,240.0,660.0,300.0,2026-09-27 00:00:00-04:00,2026-09-27 10:08:00-04:00,2026-09-27 10:08:00-04:00,2026-09-28 03:02:42-04:00,2026-09-28 02:26:25-04:00,1014.700000,2026-09-28 03:00:00-04:00


In [8]:

"""Every published service date. Use this to pick a real date range."""
resp = requests.get(INDEX_URL, headers=HEADERS, timeout=60)
resp.raise_for_status()

df = pd.read_csv(io.BytesIO(resp.content))
df.columns = [c.strip().lower() for c in df.columns]

date_col = next((c for c in df.columns if "date" in c), df.columns[0])
df["service_date"] = pd.to_datetime(df[date_col], errors="coerce")
df.dropna(subset=["service_date"]).sort_values("service_date")

,size_bytes,last_modified,service_date,file_url
0,3066,2026-01-09 21:31:47+00:00,2019-09-15,https://performancedata.mbta.com/lamp/subway-o...
1,1094250,2026-01-09 20:13:51+00:00,2019-09-16,https://performancedata.mbta.com/lamp/subway-o...
2,1520513,2026-01-09 21:39:25+00:00,2019-09-17,https://performancedata.mbta.com/lamp/subway-o...
3,1476627,2026-01-09 20:05:31+00:00,2019-09-18,https://performancedata.mbta.com/lamp/subway-o...
4,1201227,2026-01-09 22:15:38+00:00,2019-09-19,https://performancedata.mbta.com/lamp/subway-o...
...,...,...,...,...
2567,1469143,2026-09-27 00:15:02+00:00,2026-09-25,https://performancedata.mbta.com/lamp/subway-o...
2568,955096,2026-09-28 00:11:11+00:00,2026-09-26,https://performancedata.mbta.com/lamp/subway-o...
2569,872020,2026-09-29 00:17:17+00:00,2026-09-27,https://performancedata.mbta.com/lamp/subway-o...
2570,1176752,2026-09-29 21:51:45+00:00,2026-09-28,https://performancedata.mbta.com/lamp/subway-o...


In [10]:
df['file_url'][0]

'https://performancedata.mbta.com/lamp/subway-on-time-performance-v1/2019-09-15-subway-on-time-performance-v1.parquet'

In [7]:
df

,size_bytes,last_modified,service_date,file_url
0,3066,2026-01-09 21:31:47+00:00,2019-09-15,https://performancedata.mbta.com/lamp/subway-o...
1,1094250,2026-01-09 20:13:51+00:00,2019-09-16,https://performancedata.mbta.com/lamp/subway-o...
2,1520513,2026-01-09 21:39:25+00:00,2019-09-17,https://performancedata.mbta.com/lamp/subway-o...
3,1476627,2026-01-09 20:05:31+00:00,2019-09-18,https://performancedata.mbta.com/lamp/subway-o...
4,1201227,2026-01-09 22:15:38+00:00,2019-09-19,https://performancedata.mbta.com/lamp/subway-o...
...,...,...,...,...
2567,1469143,2026-09-27 00:15:02+00:00,2026-09-25,https://performancedata.mbta.com/lamp/subway-o...
2568,955096,2026-09-28 00:11:11+00:00,2026-09-26,https://performancedata.mbta.com/lamp/subway-o...
2569,872020,2026-09-29 00:17:17+00:00,2026-09-27,https://performancedata.mbta.com/lamp/subway-o...
2570,1176752,2026-09-29 21:51:45+00:00,2026-09-28,https://performancedata.mbta.com/lamp/subway-o...


In [ ]:
import io
import zipfile

import pandas as pd
import requests

GSE_URL = (
    "https://services1.arcgis.com/ceiitspzDAHrdGO1/arcgis/rest/services/"
    "GSE/FeatureServer/0/query"
)
GSE_HIST_ZIP = (
    "https://www.arcgis.com/sharing/rest/content/items/"
    "7859894afb5641ce91a2bb03599fdf5b/data"
)
HEADERS = {"User-Agent": "transit-delay-weather-class-project/1.0"}
SERVICE_TZ = "America/New_York"
PAGE = 2000  # server max per request


def _in(field, values):
    return f" AND {field} IN (" + ",".join(f"'{v}'" for v in values) + ")"


def _clean(df):
    """Shared cleanup for API and historical CSV rows."""
    df["gated_entries"] = pd.to_numeric(df["gated_entries"], errors="coerce")  # arrives as string
    # Service day runs 03:00-02:59, so 00:00-02:30 periods are the next calendar day.
    t = pd.to_timedelta(df["time_period"].astype(str).str.strip("()"))
    t = t.where(t >= pd.Timedelta(hours=3), t + pd.Timedelta(days=1))
    df["period_start"] = df["service_date"] + t
    df["hour"] = df["period_start"].dt.hour
    return df


def fetch_gse(start, end, lines=None, stations=None):
    """Raw 30-min gated entries, start <= service_date < end ('YYYY-MM-DD').

    API covers ~last 2 years (currently 2024-08-25 .. 2026-06-30).
    lines:    e.g. ["Red Line", "Green Line"]
    stations: e.g. ["place-sstat"]  (same IDs as LAMP parent_station)
    """
    where = (
        f"service_date >= TIMESTAMP '{start} 00:00:00' "
        f"AND service_date < TIMESTAMP '{end} 00:00:00'"
    )
    if lines:
        where += _in("route_or_line", lines)
    if stations:
        where += _in("stop_id", stations)

    rows, offset = [], 0
    while True:
        params = {
            "f": "json",
            "where": where,
            "outFields": "service_date,time_period,stop_id,station_name,route_or_line,gated_entries",
            "orderByFields": "ObjectId",
            "resultOffset": offset,
            "resultRecordCount": PAGE,
            "returnGeometry": "false",
        }
        resp = requests.get(GSE_URL, params=params, headers=HEADERS, timeout=120)
        resp.raise_for_status()
        data = resp.json()
        if "error" in data:
            raise RuntimeError(data["error"])
        feats = data.get("features", [])
        rows.extend(f["attributes"] for f in feats)
        if not feats or not data.get("exceededTransferLimit"):
            break
        offset += len(feats)

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["service_date"] = (
        pd.to_datetime(df["service_date"], unit="ms", utc=True)
        .dt.tz_convert(SERVICE_TZ).dt.tz_localize(None).dt.normalize()
    )
    return _clean(df)


def fetch_gse_historical(years):
    """Pre-2024 data from the yearly CSV zip (large download)."""
    resp = requests.get(GSE_HIST_ZIP, headers=HEADERS, timeout=900)
    resp.raise_for_status()
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    frames = []
    for y in years:
        with zf.open(f"GSE_by_year/GSE_{y}.csv") as fh:
            part = pd.read_csv(fh)
        part.columns = [c.strip().lower() for c in part.columns]
        part["service_date"] = pd.to_datetime(part["service_date"]).dt.tz_localize(None).dt.normalize()
        frames.append(part)
    return _clean(pd.concat(frames, ignore_index=True))


def aggregate(df, freq="W-SUN", by=("route_or_line",)):
    """freq: 'D' | 'W-SUN' (Mon-Sun weeks) | 'MS' (month)
    by:   ("route_or_line",) | ("stop_id", "station_name") | ("stop_id", "hour")
    """
    return (
        df.groupby([pd.Grouper(key="service_date", freq=freq), *by])["gated_entries"]
        .sum().reset_index()
    )


# --- usage ---
raw = fetch_gse("2026-06-01", "2026-06-29")
weekly_by_line = aggregate(raw, "W-SUN")
weekly_by_station = aggregate(raw, "W-SUN", by=("stop_id", "station_name", "route_or_line"))
hourly_by_station = aggregate(raw, "D", by=("stop_id", "hour"))  # for rider-minutes